# 03 · 3D cone beam, whole volume at once (GPU)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/diego-dupuy/ct-reconstruction-cuda/blob/main/notebooks/03_3d_conebeam.ipynb)

The whole volume is projected with a cone beam and reconstructed in a single GPU call per method.

* **ASTRA has no 3D SART**: `SART3D_CUDA` gives *"Unknown algorithm type"*. In my report I had labelled this as a memory error by mistake, so SART is not run here.
* **Detector rows**: the cone also magnifies vertically, so the detector needs more rows than the volume has slices. Otherwise the top and bottom slices are cut off. `conebeam_3d` calculates the number of rows it needs.

**Before running:** *Runtime → Change runtime type → T4 GPU*, and put `datos_ct.mat` in the root of your Google Drive.

In [ ]:
# Setup: in Colab, install ASTRA and download this repository
import os, sys
if "google.colab" in sys.modules:
    !pip install -q astra-toolbox
    if not os.path.exists("ct-reconstruction-cuda"):
        !git clone -q https://github.com/diego-dupuy/ct-reconstruction-cuda
    os.chdir("ct-reconstruction-cuda")
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")  # running locally from the notebooks/ folder
sys.path.insert(0, os.getcwd())

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from src.data import load_ct_volume
volume = load_ct_volume("/content/drive/MyDrive/datos_ct.mat")   # (512, 512, 280)
print("volume:", volume.shape)

In [ ]:
from src.reconstruction import conebeam_3d

methods = {"BP": 1, "FDK": 1, "SIRT": 20, "CGLS": 10}
images, times = conebeam_3d(volume, methods)
times

In [ ]:
import pandas as pd
from src.metrics import evaluate

# BP is not normalised by ASTRA (arbitrary scale), so it is left out of the metrics
table = pd.DataFrame(evaluate(volume, {k: v for k, v in images.items() if k != "BP"})).T
table["GPU time (s)"] = pd.Series(times)
table.round(3)

In [ ]:
FIGURE = "docs/figures/conebeam_slice.png"
import matplotlib.pyplot as plt
from src.visualization import show_comparison

z = volume.shape[2] // 2
central = {name: img[:, :, z] for name, img in images.items()}
fig = show_comparison(volume[:, :, z], central, times=times, ncols=3,
                      title=f"Central axial slice (z = {z})")
fig.savefig(FIGURE, dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
import numpy as np
from src.metrics import fov_mask
from src.visualization import show_isosurface

BONE_LEVEL = 0.5   # bone is above ~0.5 in the normalised volume (see the slices above)
bones = images["CGLS"].copy()
bones[~fov_mask(bones.shape)] = 0   # remove streaks outside the scanned circle
fig = show_isosurface(bones, level=BONE_LEVEL, step_size=2, title="Bone isosurface (CGLS)")
plt.show()